In [2]:
#---Core---
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

#---Visualization---
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

#---Preprocessing---
from sklearn.model_selection import (train_test_split, StratifiedKFold, 
                                    GridSearchCV, RandomizedSearchCV,
                                    cross_val_score)
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

#---Models---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

#---Evaluation---
from sklearn.metrics import (accuracy_score, classification_report,
                            confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve)

#---Settings---
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid', palette='muted')

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
#---Load Data---
df = pd.read_csv('combined_output.csv')

#---Basic Shape ---
print("Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

#---Source Files Present---
print("\nUnique Source Files:")
print(df['_source_file'].value_counts())

#---Preview---
print("\nData Preview:")
print(df.head(10))

Shape: (136352, 101)

Column Names:
['Player', 'Player ID', '_source_file', 'Abbreviated', 'Full Name', 'Tournament', 'Stage', 'Match Type', 'Match Name', 'Map', 'Round Number', 'Team', 'Loadout Value', 'Remaining Credits', 'Type', 'Outcome', 'Tournament ID', 'Stage ID', 'Match Type ID', 'Agent', 'Total Wins By Map', 'Total Loss By Map', 'Total Maps Played', 'Team ID', 'Initiated', 'Won', 'Action', 'Pick Rate', 'Method', 'Agents', 'Rating', 'Average Combat Score', 'Kills', 'Deaths', 'Assists', 'Kills - Deaths (KD)', 'Kill, Assist, Trade, Survive %', 'Average Damage Per Round', 'Headshot %', 'First Kills', 'First Deaths', 'Kills - Deaths (FKD)', 'Side', 'Teams', 'Rounds Played', 'Kills:Deaths', 'Kills Per Round', 'Assists Per Round', 'First Kills Per Round', 'First Deaths Per Round', 'Clutch Success %', 'Clutches (won/played)', 'Maximum Kills in a Single Map', 'Team A', 'Team B', 'Team A Score', 'Team B Score', 'Match Result', 'Match ID', 'Game ID', 'Elimination', 'Detonated', 'Defused'

In [6]:
#Split combined dataframe into individual source dataframes
source_files = df['_source_file'].unique()

#Create a dictionary to hold the individual dataframes
dfs = {source: df[df['_source_file'] == source].reset_index(drop=True) 
       for source in source_files}

#Preview the most important ones
for name in ['players_stats.csv', 'overview.csv', 'scores.csv',
             'maps_scores.csv', 'maps_stats.csv']:
    print(f"\n{'='*50}")
    print(f"SOURCE: {name}, SHAPE: {dfs[name].shape}")
    #Only show columns that are not all null
    non_null_cols = dfs[name].columns[dfs[name].notna().any()].tolist()
    print(f"Non-null columns: {non_null_cols}")
    print(dfs[name][non_null_cols].head(3))



SOURCE: players_stats.csv, SHAPE: (5231, 101)
Non-null columns: ['Player', '_source_file', 'Tournament', 'Stage', 'Match Type', 'Agents', 'Rating', 'Average Combat Score', 'Kills', 'Deaths', 'Assists', 'Kill, Assist, Trade, Survive %', 'Average Damage Per Round', 'Headshot %', 'First Kills', 'First Deaths', 'Teams', 'Rounds Played', 'Kills:Deaths', 'Kills Per Round', 'Assists Per Round', 'First Kills Per Round', 'First Deaths Per Round', 'Clutch Success %', 'Clutches (won/played)', 'Maximum Kills in a Single Map']
  Player       _source_file                      Tournament     Stage  \
0  skuba  players_stats.csv  Valorant Masters Santiago 2026  Playoffs   
1  skuba  players_stats.csv  Valorant Masters Santiago 2026  Playoffs   
2  skuba  players_stats.csv  Valorant Masters Santiago 2026  Playoffs   

            Match Type        Agents  Rating  Average Combat Score  Kills  \
0  Upper Quarterfinals         astra    1.93                 289.0   22.0   
1  Upper Quarterfinals         v